In [ ]:
%load_ext jupyter_black

In [ ]:
import warnings

warnings.filterwarnings("ignore")

In [ ]:
import os
import gc
from dotenv import load_dotenv

import torch
import numpy as np
import pandas as pd
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
from sentence_transformers import SentenceTransformer

In [ ]:
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
client = QdrantClient("localhost", port=6333)

load_dotenv()
hf_token = os.getenv("HF_TOKEN")
model_name = "Qwen/Qwen3-Embedding-0.6B"
model = SentenceTransformer(model_name, token=hf_token)

In [ ]:
if not client.collection_exists("legal_docs"):
    client.create_collection(
        collection_name="legal_docs",
        vectors_config=VectorParams(size=768, distance=Distance.COSINE),
    )

In [ ]:
chunks = pd.read_json("../data/processed/chunks/trafik_kanunu.jsonl", lines=True)

In [ ]:
def embed_in_batches(model, texts, batch_size=4):
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        with torch.no_grad():
            embeddings = model.encode(batch, convert_to_numpy=True)
        all_embeddings.append(embeddings)
        torch.mps.empty_cache()

    # Aggressive cleanup after each batch
    torch.cuda.empty_cache()
    gc.collect()

    return np.vstack(all_embeddings)

In [ ]:
# Convert to lists once
texts = chunks["text"].tolist()
metadatas = chunks["metadata"].tolist()

# Embed in batches
embeddings = embed_in_batches(model, texts, batch_size=8)

# Build points
points = [
    PointStruct(id=i, vector=emb.tolist(), payload=chunk)
    for i, (chunk, emb) in enumerate(zip(metadatas, embeddings))
]

# Upsert in batches too (avoids large payloads)
UPSERT_BATCH = 100
for i in range(0, len(points), UPSERT_BATCH):
    client.upsert(collection_name="legal_docs", points=points[i : i + UPSERT_BATCH])

In [ ]:
batch_size = 5
all_embeddings = []
for i in range(0, len(chunks), batch_size):
    batch = chunks[i:i+batch_size]
    with torch.no_grad():
        embeddings = model.encode(batch)  # or model.embedding(batch)
    all_embeddings.append(embeddings)
    torch.mps.empty_cache()  # Clear MPS cache on Apple Silicon